# Summarization & Entity Recognition

This notebook builds structured product insights from reviews and metadata, then exports a unified profile file for downstream semantic search and demo notebooks.

It supports the objective of extracting actionable product intelligence from customer feedback using entity extraction and concise summaries.

> **Prerequisite:** notebook 04 must be run first — the final merge step reads `data/embedding_index_enriched.csv` produced there.


## Setup

Load dependencies, configuration paths, and helper utilities required for entity extraction and summarization.


In [ ]:
import os
import sys

SRC_DIR = os.path.join(os.getcwd(), "src")
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from mean_squared_terrors.summarization import setup_notebook_dependencies

setup_notebook_dependencies()


In [ ]:
import os
import warnings

import pandas as pd

from mean_squared_terrors.summarization import get_device, load_clean_data

warnings.filterwarnings("ignore")

DATA_DIR = os.path.join(os.getcwd(), "data")
products, reviews = load_clean_data(DATA_DIR)
DEVICE = get_device()

print(f"Prodotti: {len(products):,}  |  Review: {len(reviews):,}")
print(f"Device per transformers: {'GPU' if DEVICE == 0 else 'CPU'}")


## Entity Recognition

Extract structured attributes from product text and reviews:

1. Ingredients (regex-based patterns)
2. Product concerns and use-cases
3. Suitable skin types


In [ ]:
import spacy

from mean_squared_terrors.summarization import build_entities_dataframe

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

entities_df = build_entities_dataframe(products, reviews, nlp)
print(f"\nEntities extracted for {len(entities_df):,} products")
print(f"With at least 1 ingredient:     {(entities_df['n_ingredients']>0).mean():.1%}")
print(f"With at least 1 certification:  {(entities_df['n_certifications']>0).mean():.1%}")
print(f"With at least 1 use case:        {(entities_df['use_cases']!='').mean():.1%}")

entities_df.to_csv(os.path.join(DATA_DIR, "product_entities.csv"), index=False)
print("\nSaved product_entities.csv")

print("\n--- 3 examples ---")
for _, r in entities_df[entities_df['n_ingredients'] > 2].head(3).iterrows():
    print(f"\n{r['product_title'][:90]}")
    print(f"  ingredients: {r['ingredients']}")
    print(f"  certifications: {r['certifications']}")
    print(f"  use_cases: {r['use_cases']}")


In [ ]:
# Entity extraction output already produced in previous cell.
# This cell is intentionally kept minimal to preserve notebook section flow.
print("Entity extraction stage completed.")


## Review Summarization

Generate concise positive and negative summaries per product with a two-stage process:

1. Extract informative sentences from review buckets
2. Produce final summaries with the summarization model


In [ ]:
from mean_squared_terrors.summarization import build_summarizer

summarizer = build_summarizer(DEVICE)


In [ ]:
from mean_squared_terrors.summarization import run_batch_summarization

print("Batch summarization helper imported.")


In [ ]:
print("Starting batch summarization...")
print("Estimated time: long run depending on hardware and model cache.")
print("Press Interrupt to stop — partial results are saved every 500 products.\n")

summaries_df = run_batch_summarization(
    products=products,
    reviews=reviews,
    summarizer=summarizer,
    data_dir=DATA_DIR,
    save_every=500,
)
summaries_df.to_csv(os.path.join(DATA_DIR, "product_summaries.csv"), index=False)

print(f"Products with BART summary:    {(summaries_df['method']=='model').sum():,}")
print(f"Products with extractive:      {(summaries_df['method']=='extractive').sum():,}")
print("Saved product_summaries.csv")


## Quality Check

Validate output coverage and inspect summary/entity quality before exporting final artifacts.


In [ ]:
if 'summaries_df' not in dir() or len(summaries_df) == 0:
    summaries_df = pd.read_csv(os.path.join(DATA_DIR, "product_summaries.csv"))
    print(f"Reloaded: {len(summaries_df):,} summaries")

print(f"Total summaries: {len(summaries_df):,}")
print(f"With non-empty summary: {(summaries_df['summary_full']!='').mean():.1%}")
print(f"With pros: {(summaries_df['pros']!='').mean():.1%}")
print(f"With cons: {(summaries_df['cons']!='').mean():.1%}")
print(f"BART method: {(summaries_df['method']=='model').mean():.1%}")

print("\n--- Sample summaries (BART method) ---")
model_summs = summaries_df[
    (summaries_df['method'] == 'model')
    & (summaries_df['pros'] != '')
    & (summaries_df['cons'] != '')
].head(5)

for _, r in model_summs.iterrows():
    prod_title = products[products['parent_asin'] == r['parent_asin']]['product_title'].values
    title = prod_title[0][:60] if len(prod_title) > 0 else r['parent_asin']
    print(f"\n{title}")
    print(f"  Summary: {r['summary_full']}")
    print(f"  Pros:    {r['pros'][:120]}")
    print(f"  Cons:    {r['cons'][:120]}")


## Merge with Embedding Index

Combine summaries and extracted entities into `product_profiles.csv`, the profile table used by semantic search and demo notebooks.


In [ ]:
from mean_squared_terrors.summarization import build_product_profiles

idx = build_product_profiles(DATA_DIR, summaries_df)
idx.to_csv(os.path.join(DATA_DIR, "product_profiles.csv"), index=False)

print(f"product_profiles.csv saved: {len(idx):,} rows")
print(f"Columns: {list(idx.columns)}")
print("\nCoverage:")
print(f"  With summary:      {idx['summary_full'].notna().mean():.1%}")
print(f"  With pros:         {(idx['pros'].fillna('')!='').mean():.1%}")
print(f"  With ingredients:  {(idx['ingredients'].fillna('')!='').mean():.1%}")
print(f"  With certifications: {(idx['certifications'].fillna('')!='').mean():.1%}")
